# Notebook 12: CUDA Graph 机制与对 GEMM 的影响

## 本章内容

前面几章我们一直在讲怎么写 Triton kernel、怎么替换算子。但在 vLLM 的生产环境中，模型推理并不是"裸跑" kernel 的——中间还有一层极其重要的加速机制：**CUDA Graph**。

如果你不理解 CUDA Graph，就无法回答这些关键问题：
- 为什么 vLLM 启动时要花很长时间 "Capturing CUDA graphs"？
- `FULL_DECODE_ONLY` 和 `PIECEWISE` 到底是什么意思？
- 我写的自定义 Triton GEMM 放进去后，M/N/K 是编译时常量还是运行时动态的？
- Padding 对 GEMM 的性能影响有多大？

本章从零开始，把 CUDA Graph 的机制讲透。

> **前置知识：** Notebook 01（Triton 基础），Notebook 06（GEMM 全景），Notebook 11（自定义 GEMM 部署）

## 1. 为什么需要 CUDA Graph

### 1.1 GPU 推理的隐藏瓶颈：CPU Launch Overhead

GPU 上跑一个 kernel 的流程是这样的：

```
CPU 端                                GPU 端
──────                                ──────
1. 准备 kernel 参数
2. 调用 cudaLaunchKernel()   ─────▸  3. 开始执行 kernel
                                      4. 执行完毕
5. 准备下一个 kernel 参数
6. 调用 cudaLaunchKernel()   ─────▸  7. 开始执行下一个 kernel
    ...                                ...
```

每次 `cudaLaunchKernel()` 都有 **5-10 微秒** 的 CPU 端开销。

一个 Transformer 层有多少 kernel？粗略数一下：

| 操作 | kernel 数量 |
|------|------------|
| LayerNorm | 1 |
| QKV GEMM | 1 |
| RoPE | 1 |
| Attention | 2-3 |
| O GEMM | 1 |
| LayerNorm | 1 |
| Gate+Up GEMM | 1 |
| SiLU+Mul | 1 |
| Down GEMM | 1 |
| Add (residual) | 1 |
| **合计** | **~11** |

一个 32 层模型 = **~352 个 kernel**。每个 5μs → **1.76ms 纯 CPU 开销**。

问题在于：**Decode 阶段每次只生成 1 个 token**，GEMM 本身可能只需要几十微秒（M=1 的 GEMV），但 CPU launch 就要 5μs。当 kernel 计算时间和 launch 时间差不多大时，CPU 就成了瓶颈。

### 1.2 CUDA Graph：把 kernel 提交"录像"下来

CUDA Graph 的思路非常简单：

```
录制阶段 (Capture):
  CPU: "我要启动 kernel A → kernel B → kernel C"
  CUDA: "好的，我把这个序列记下来了"

回放阶段 (Replay):
  CPU: graph.replay()     ← 一次 API 调用
  GPU: A → B → C          ← 直接回放整个序列
```

从 ~352 次 `cudaLaunchKernel()` 变成 **1 次 `graph.replay()`**。CPU 开销从 1.76ms 降到 ~10μs。

In [ ]:
import torch
import time

# ============================================================
# 演示 CUDA Graph 的加速效果
# ============================================================

def create_simple_model_ops(x, weights):
    """模拟一个简化的 Transformer 层"""
    # LayerNorm (简化)
    x = x / (x.norm(dim=-1, keepdim=True) + 1e-6)
    # GEMM 1 (QKV projection)
    x = torch.nn.functional.linear(x, weights[0])
    # GEMM 2 (O projection)
    x = torch.nn.functional.linear(x, weights[1])
    # Activation
    x = torch.nn.functional.silu(x)
    # GEMM 3 (Down projection)
    x = torch.nn.functional.linear(x, weights[2])
    return x


# 准备数据
device = 'cuda'
M, K = 32, 1024
weights = [
    torch.randn(K, K, device=device, dtype=torch.float16) for _ in range(3)
]
x = torch.randn(M, K, device=device, dtype=torch.float16)

# --- Eager 模式 ---
# warmup
for _ in range(10):
    _ = create_simple_model_ops(x, weights)
torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(1000):
    _ = create_simple_model_ops(x, weights)
torch.cuda.synchronize()
eager_time = (time.perf_counter() - start) / 1000

# --- CUDA Graph 模式 ---
# 1. 录制
graph = torch.cuda.CUDAGraph()
# 使用静态输入（CUDA Graph 要求输入地址不变）
static_x = x.clone()

with torch.cuda.graph(graph):
    static_out = create_simple_model_ops(static_x, weights)

# 2. 回放
# warmup
for _ in range(10):
    static_x.copy_(x)
    graph.replay()
torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(1000):
    static_x.copy_(x)  # 只需要 copy 数据到静态缓冲区
    graph.replay()       # 一次 API 调用回放所有 kernel
torch.cuda.synchronize()
graph_time = (time.perf_counter() - start) / 1000

speedup = eager_time / graph_time
print(f"Eager 模式:      {eager_time*1e6:.1f} us/iter")
print(f"CUDA Graph 模式: {graph_time*1e6:.1f} us/iter")
print(f"加速比:          {speedup:.2f}x")
print()
print("注意：batch 越小，加速越明显（因为 kernel 计算时间短，launch overhead 占比高）")

## 2. CUDA Graph 的硬约束

CUDA Graph 不是免费午餐，它有严格的限制：

### 约束 1: 张量形状必须固定

```
录制时: input.shape = (32, 4096)  →  GEMM kernel 的参数是 M=32, K=4096, N=11008
回放时: input.shape 必须还是 (32, 4096)
       如果变成 (33, 4096) → 行为未定义！（通常 silent corruption）
```

这是最关键的约束。推理时 batch size 是动态的（每个请求的 token 数不同），但 CUDA Graph 要求固定形状。

### 约束 2: 不能包含 CPU-GPU 同步

```
❌ 不能有 if tensor.item() > 0:   (需要 CPU 读 GPU 值)
❌ 不能有 torch.cuda.synchronize()
❌ 不能有动态 shape 的操作（如 torch.nonzero()）
```

### 约束 3: 内存地址必须固定

录制时使用的 GPU 内存地址，在回放时必须是同一块。所以需要预分配"静态缓冲区"。

### 约束 4: 不能跨 stream 录制

一个 CUDA Graph 在一个 stream 上录制，回放也在同一个 stream 上。

### 这些约束对 GEMM 意味着什么

| 约束 | 对 GEMM 的影响 |
|------|---------------|
| 形状固定 | M 必须在录制时确定，不能动态变化 |
| 无 CPU 同步 | GEMM 本身没有同步，不受影响 |
| 地址固定 | weight 地址固定（没问题），input 需要用静态缓冲区 |
| 单 stream | GEMM 天然在单 stream，不受影响 |

说白了，**CUDA Graph 对 GEMM 的唯一影响就是 M 必须固定**。

## 3. vLLM 的 CUDA Graph 模式

vLLM 提供了多种 CUDA Graph 策略，通过 `--compilation-config.cudagraph_mode` 配置：

| 模式 | 值 | 含义 |
|------|---|------|
| `NONE` | 0 | 不使用 CUDA Graph |
| `PIECEWISE` | 1 | 分段录制：把 Attention 等不兼容的 op 排除在外 |
| `FULL` | 2 | 全图录制：整个模型 forward 作为一个 graph |
| `FULL_DECODE_ONLY` | (2, 0) | Decode 用 FULL，Prefill 不用 graph |
| `FULL_AND_PIECEWISE` | (2, 1) | Decode 用 FULL，Prefill 用 PIECEWISE（**默认值**） |

### 为什么要区分 Decode 和 Prefill？

```
Decode 阶段:
  每个请求只生成 1 个 token → num_tokens = num_requests
  M 很小 (1~512) → kernel 计算时间短 → launch overhead 占比高
  → CUDA Graph 收益巨大

Prefill 阶段:
  一次处理整个 prompt → num_tokens 可能很大 (512~8192)
  M 很大 → kernel 计算时间长 → launch overhead 占比低
  → CUDA Graph 收益小，而且大 M 的 graph 占显存多
```

这就是 `FULL_DECODE_ONLY` 的设计思路：只在最需要的地方用 CUDA Graph。

> Source: `vllm/config/compilation.py:51-61`（CUDAGraphMode 枚举定义）

### 3.1 PIECEWISE vs FULL 的区别

某些操作和 CUDA Graph 不兼容（主要是 Attention kernel），vLLM 用两种策略处理：

**FULL 模式：**
```
┌─────────────────────────────────────────────────┐
│              整个 forward 作为一个 graph            │
│  LayerNorm → QKV → Attn → O → LN → MLP → ...   │
└─────────────────────────────────────────────────┘
要求 Attention backend 必须支持 CUDA Graph
```

**PIECEWISE 模式：**
```
┌──────────────┐         ┌──────────────┐
│  Graph 片段 1  │  Attn   │  Graph 片段 2  │  Attn  ...
│  LN → QKV    │ (eager) │  O → LN → MLP│ (eager)
└──────────────┘         └──────────────┘
把 Attention 排除在 graph 之外，其余部分分段录制
```

**FULL_AND_PIECEWISE (默认)：**
```
Decode 批次 → FULL graph (Attention 也能 graph 化)
混合/Prefill 批次 → PIECEWISE graph (Attention 用 eager)
```

> Source: `vllm/v1/worker/gpu/cudagraph_utils.py:252-273`（两阶段 capture）

## 4. Capture Sizes：Pad-to-Captured-Size 策略

### 4.1 核心问题

CUDA Graph 要求形状固定，但 batch size 是动态的。怎么办？

vLLM 的方案：**预录制一组离散的 batch size，运行时 pad 到最近的录制尺寸。**

### 4.2 默认 capture sizes

vLLM 自动计算录制尺寸：

```python
# vllm/config/vllm.py:1269-1281
capture_sizes = [1, 2, 4]             # 小 batch: 精确
capture_sizes += range(8, 256, 8)     # 中 batch: 步长 8
capture_sizes += range(256, max+1, 16) # 大 batch: 步长 16
```

假设 max=512，完整列表是：
```
[1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96,
 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192,
 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320,
 336, 352, 368, 384, 400, 416, 432, 448, 464, 480, 496, 512]
```

**约 50 个 capture size**，每个都需要录制一个完整的 CUDA Graph。

In [ ]:
# ============================================================
# 模拟 vLLM 的 capture size 计算
# ============================================================

def compute_capture_sizes(max_size=512):
    """复现 vLLM 的默认 capture sizes 计算逻辑"""
    sizes = [i for i in [1, 2, 4] if i <= max_size]
    if max_size >= 8:
        sizes += list(range(8, min(max_size + 1, 256), 8))
    if max_size >= 256:
        sizes += list(range(256, max_size + 1, 16))
    return sorted(set(sizes))


def build_padding_map(capture_sizes):
    """复现 vLLM 的 padding 映射逻辑"""
    padding_map = {}
    for i in range(1, capture_sizes[-1] + 1):
        for x in capture_sizes:
            if i <= x:
                padding_map[i] = x
                break
    return padding_map


# 计算
capture_sizes = compute_capture_sizes(512)
padding_map = build_padding_map(capture_sizes)

print(f"Capture sizes 数量: {len(capture_sizes)}")
print(f"Capture sizes: {capture_sizes}")
print()

# 展示一些 padding 示例
examples = [1, 2, 3, 5, 7, 9, 13, 31, 33, 64, 100, 127, 200, 255, 257, 400, 500, 512]
print(f"{'实际batch':>10} {'Pad到':>8} {'浪费':>8} {'浪费率':>8}")
print("-" * 40)
for actual in examples:
    padded = padding_map.get(actual, None)
    if padded is None:
        print(f"{actual:>10} {'超出范围':>8}")
    else:
        waste = padded - actual
        waste_pct = waste / padded * 100
        print(f"{actual:>10} {padded:>8} {waste:>8} {waste_pct:>7.1f}%")

## 5. CUDA Graph 对 GEMM M/N/K 维度的影响

这是本章最核心的问题。让我们追踪一次 Decode 推理中 MLP GEMM 的 M/N/K：

### 5.1 gate_up_proj GEMM

```
gate_up_proj: Y = X @ W^T
             (M, K)   (K, N) → (M, N)

M = num_tokens_after_padding   ← CUDA Graph 决定！
K = hidden_size = 4096         ← 模型结构决定，编译时常量
N = intermediate_size × 2 = 22016  ← 模型结构决定，编译时常量
```

### 5.2 三个维度的性质

| 维度 | 来源 | 是否常量 | CUDA Graph 影响 |
|------|------|---------|----------------|
| **K** | `hidden_size` | ✅ 编译时常量 | 无影响 |
| **N** | `intermediate_size` | ✅ 编译时常量 | 无影响 |
| **M** | `num_tokens` | ❌ **运行时动态** | 被约束到 ~50 个离散值 |

### 5.3 M 不是编译时常量！

很多人会误以为 CUDA Graph 让 M 变成了常量。**不是的。**

```
torch.compile + @support_torch_compile(dynamic_arg_dims={"input_ids": 0})
                                                         ▲
                                                    第 0 维是动态的
```

`torch.compile` 把 M 标记为 **symbolic dynamic**，生成的 Inductor Triton kernel 会把 M 作为运行时参数：

```python
# Inductor 生成的 kernel 类似于：
def triton_kernel(..., M, ...):  # M 是运行时传入的
    ...
```

**CUDA Graph 的作用是：每个 capture size 对应一个录制好的 graph。** 当 batch=5 时，pad 到 8，使用为 M=8 录制的 graph。这个 graph 里面的 kernel 参数就是 M=8。

所以准确地说：**M 不是编译时常量，但在每次 replay 时是固定的（= capture size）。**

> Source: `vllm/model_executor/models/qwen3_5.py:290-298`（dynamic_arg_dims）
> Source: `vllm/compilation/decorators.py:137-159`（dynamic_arg_dims 文档）

### 5.4 Padding 对 GEMM 计算量的浪费

In [ ]:
# ============================================================
# 计算不同 batch size 下 padding 导致的 GEMM 计算浪费
# ============================================================

capture_sizes = compute_capture_sizes(512)
padding_map = build_padding_map(capture_sizes)

K, N = 4096, 22016  # Qwen3-8B MLP gate_up_proj

print("=" * 70)
print("Qwen3-8B MLP gate_up_proj: Y = X @ W^T,  K=4096, N=22016")
print("=" * 70)
print()

# 统计不同区间的平均浪费率
ranges = [(1, 8), (8, 64), (64, 256), (256, 512)]
for lo, hi in ranges:
    waste_pcts = []
    for actual in range(lo, hi + 1):
        padded = padding_map.get(actual, actual)
        waste_pcts.append((padded - actual) / padded * 100)
    avg_waste = sum(waste_pcts) / len(waste_pcts)
    max_waste = max(waste_pcts)
    # 计算多余的 FLOPs
    # GEMM FLOPs = 2 * M * N * K
    worst_actual = lo
    worst_padded = padding_map.get(worst_actual, worst_actual)
    extra_flops = 2 * (worst_padded - worst_actual) * N * K / 1e9
    print(f"M 在 [{lo:>3}, {hi:>3}] 范围:")
    print(f"  平均浪费率: {avg_waste:.1f}%")
    print(f"  最大浪费率: {max_waste:.1f}% (M={lo} pad到 {padding_map.get(lo, lo)})")
    print(f"  最差情况多余 FLOPs: {extra_flops:.2f} GFLOPS")
    print()

print("结论:")
print("1. 小 batch (M<8) 浪费率最高，但绝对 FLOPs 很小")
print("2. 大 batch (M>64) 浪费率 < 10%，基本可忽略")
print("3. GEMM 是 compute-bound 的，多算几行的代价远小于 CUDA Graph 省下的 launch overhead")

## 6. FULL_DECODE_ONLY 模式详解

这是用户提问的焦点。让我们完整追踪一次 FULL_DECODE_ONLY 模式下的推理流程。

### 6.1 模式定义

```python
# vllm/config/compilation.py:60
FULL_DECODE_ONLY = (FULL, NONE)
#                    ▲      ▲
#               decode 用 FULL   prefill 不用 graph
```

### 6.2 运行时分发逻辑

```python
# vllm/v1/worker/gpu/cudagraph_utils.py:275-289
def get_cudagraph_runtime_mode(self, num_reqs, num_tokens, max_query_len):
    # 判断是否是纯 decode 批次
    is_uniform_decode = (max_query_len == 1) and (num_tokens == num_reqs)

    cudagraph_size = self.get_cudagraph_size(num_tokens, is_uniform_decode)

    if cudagraph_size is None:
        return CUDAGraphMode.NONE     # 超出 capture range，不用 graph
    elif is_uniform_decode:
        return self.cudagraph_mode.decode_mode()  # → FULL
    else:
        return self.cudagraph_mode.mixed_mode()   # → NONE (不用 graph)
```

### 6.3 不同批次类型的处理

| 批次类型 | 条件 | CUDA Graph | GEMM 的 M |
|---------|------|-----------|-----------|
| 纯 Decode | max_query_len=1, tokens=reqs | FULL graph | pad 到 capture size |
| 混合 Prefill+Decode | max_query_len>1 | 不用 graph (eager) | 完全动态 |
| 纯 Prefill | max_query_len=prompt_len | 不用 graph (eager) | 完全动态 |

> Source: `vllm/v1/worker/gpu/cudagraph_utils.py:275-289`

## 7. 录制与回放的完整流程

### 7.1 启动时的录制阶段

当 vLLM 启动并完成模型加载后，进入 CUDA Graph 录制阶段：

```
启动日志: "Capturing CUDA graphs (decode, FULL)"

for size in [512, 496, 480, ..., 8, 4, 2, 1]:   # 从大到小
    1. 准备假输入: input_ids[:size], positions[:size]
    2. Warmup: model(input_ids, positions)          # 一次 eager 执行
    3. 录制:
       graph = torch.cuda.CUDAGraph()
       with torch.cuda.graph(graph):
           hidden_states = model(input_ids, positions)
       graphs[size] = graph                          # 保存 graph
```

**关键细节：**
- 输入使用预分配的**静态缓冲区** (`input_buffers`)，地址不变
- 每个 capture size 都有自己的 CUDA Graph 实例
- 从大到小录制，这样 GPU 内存分配可以复用（`graph_pool_handle`）

### 7.2 推理时的回放阶段

```
收到 batch: 实际 5 个 token

1. 查表: padding_map[5] = 8
2. Pad: input_ids[:8] (后 3 个是 padding token)
3. 复制数据到静态缓冲区: static_input[:8] = padded_input
4. 回放: graphs[8].replay()
5. 截断: output = hidden_states[:5]   ← 丢弃 padding 部分
```

> Source: `vllm/v1/worker/gpu/cudagraph_utils.py:158-193`（capture）
> Source: `vllm/v1/worker/gpu/cudagraph_utils.py:291-295`（replay + 截断）

In [ ]:
# ============================================================
# 模拟 CUDA Graph 录制与回放的完整流程
# ============================================================

import torch

class SimpleCudaGraphManager:
    """简化版的 vLLM CudaGraphManager"""

    def __init__(self, model_fn, input_size, capture_sizes):
        self.model_fn = model_fn
        self.input_size = input_size
        self.graphs = {}
        self.outputs = {}

        # 静态输入缓冲区
        max_size = max(capture_sizes)
        self.static_input = torch.zeros(max_size, input_size,
                                         device='cuda', dtype=torch.float16)
        self.static_output = torch.zeros(max_size, input_size,
                                          device='cuda', dtype=torch.float16)

        # 构建 padding map
        self.padding_map = {}
        sorted_sizes = sorted(capture_sizes)
        for i in range(1, sorted_sizes[-1] + 1):
            for s in sorted_sizes:
                if i <= s:
                    self.padding_map[i] = s
                    break

    def capture(self, sizes):
        """录制 CUDA Graph"""
        for size in sorted(sizes, reverse=True):
            inp = self.static_input[:size]

            # Warmup
            out = self.model_fn(inp)

            # 录制
            graph = torch.cuda.CUDAGraph()
            with torch.cuda.graph(graph):
                out = self.model_fn(inp)
            self.graphs[size] = graph
            self.outputs[size] = out
            print(f"  录制 size={size}: graph captured, output.shape={out.shape}")

    def run(self, x):
        """运行推理（自动 pad + replay + 截断）"""
        actual_size = x.shape[0]
        padded_size = self.padding_map.get(actual_size)

        if padded_size is None or padded_size not in self.graphs:
            # Fallback to eager
            return self.model_fn(x), "eager"

        # 1. 复制到静态缓冲区
        self.static_input[:actual_size].copy_(x)
        self.static_input[actual_size:padded_size].zero_()  # padding 部分清零

        # 2. Replay
        self.graphs[padded_size].replay()

        # 3. 截断
        result = self.outputs[padded_size][:actual_size].clone()
        return result, f"graph(pad={padded_size})"


# 创建简单模型
weight = torch.randn(1024, 1024, device='cuda', dtype=torch.float16)
def simple_model(x):
    return torch.nn.functional.linear(x, weight)

# 录制
manager = SimpleCudaGraphManager(simple_model, 1024, [1, 2, 4, 8, 16, 32])
print("=== 录制阶段 ===")
manager.capture([1, 2, 4, 8, 16, 32])
print()

# 推理
print("=== 推理阶段 ===")
for batch_size in [1, 3, 5, 8, 16, 20, 32, 50]:
    x = torch.randn(batch_size, 1024, device='cuda', dtype=torch.float16)
    result, mode = manager.run(x)
    print(f"  batch={batch_size:>3}: mode={mode:<20} output.shape={result.shape}")

## 8. CUDA Graph 与 torch.compile 的协作

CUDA Graph 和 torch.compile 是两个独立的优化，但在 vLLM 中它们紧密配合：

### 8.1 两者的分工

```
torch.compile:
  - 分析 Python 代码，生成优化后的 kernel
  - 做算子融合（fuse LayerNorm + GEMM 等）
  - 处理动态 shape（symbolic shapes）
  - 生成 Inductor Triton kernel 或调用 cuBLAS

CUDA Graph:
  - 录制 kernel 执行序列
  - 消除 CPU launch overhead
  - 要求固定形状
```

### 8.2 PIECEWISE 模式下的协作

```
torch.compile 先编译模型:
  ┌──────────────────────┐   ┌──────────────────────┐
  │ Compiled Subgraph 1  │   │ Compiled Subgraph 2  │
  │ (LN → QKV fused)    │   │ (O → LN → MLP fused)│
  └──────────────────────┘   └──────────────────────┘
           ↓                          ↓
CUDA Graph 再分段录制:
  ┌──────────────────────┐   ┌──────────────────────┐
  │  CG Partition 1      │   │  CG Partition 2      │
  │  (compiled kernels)  │   │  (compiled kernels)  │
  └──────────────────────┘   └──────────────────────┘
           ↓ Attention (eager) ↓
```

### 8.3 Inductor 对 GEMM 的处理

当 torch.compile 遇到 `torch.nn.functional.linear` 时：
- **cuBLAS 路径**（默认）：Inductor 直接调用 cuBLAS 的 GEMM。M 作为运行时参数传入。
- **Triton codegen 路径**（当 CustomOp disabled 时）：Inductor 生成自己的 Triton GEMM kernel，M 是 symbolic dynamic。

两种情况下 **N 和 K 都是编译时常量**（它们由模型权重形状决定，在 trace 时就确定了），**只有 M 是 dynamic**。

> Source: `vllm/compilation/decorators.py:264-270`（_support_torch_compile）
> Source: `vllm/compilation/wrapper.py:1-80`（TorchCompileWithNoGuardsWrapper）

## 9. 对自定义 Triton GEMM 的影响

如果你按 Notebook 11 的方式替换了 GEMM，在 CUDA Graph 模式下需要关注以下几点：

### 9.1 Triton autotune 与 CUDA Graph

```python
@triton.autotune(
    configs=[...],
    key=['M', 'N', 'K'],  # 每组 (M, N, K) 缓存最优配置
)
@triton.jit
def our_matmul_kernel(...):
    ...
```

**好消息：** CUDA Graph 把 M 限制到了 ~50 个 capture sizes，所以 `@triton.autotune` 只需要为 ~50 种 M 值各跑一次 benchmark。之后每种 M 的最优配置都能被稳定复用。

对比没有 CUDA Graph 时：M 可以是任意正整数，autotune 需要不断为新的 M 值重新选择配置。

### 9.2 JIT 编译的时机

```
CUDA Graph 录制阶段:
  for size in capture_sizes:
    warmup(size)           ← 第一次调用，触发 Triton JIT 编译
    capture(size)          ← 第二次调用，已编译好，纯录制

关键: warmup 确保 JIT 编译在录制之前完成！
```

vLLM 的 `capture_graph()` 方法（第 128-141 行）在录制前会先做一次 warmup，确保所有 kernel 都已编译好。如果你的 Triton kernel 在 warmup 阶段就被调用，它的 JIT 编译会在那时完成，不会影响录制。

### 9.3 潜在陷阱

| 陷阱 | 原因 | 解决方案 |
|------|------|---------|
| Triton kernel 在 capture 中报错 | `autotune` 的 benchmark 需要 CPU 同步 | warmup 阶段完成 autotune |
| 不同 M 用了不同 config | autotune 为每个 M 选不同的 BLOCK_M | 这是正常行为，不影响 CUDA Graph |
| weight 形状变化 | 模型权重加载后不会变 | 不用担心 |
| `make_block_ptr` 不兼容 | 某些 Triton 版本的 block_ptr 和 CUDA Graph 有冲突 | 使用显式 pointer arithmetic |

> Source: `vllm/v1/worker/gpu/cudagraph_utils.py:128-156`（warmup + capture 流程）

## 10. 隐藏的性能收益：cuBLAS 算法选择缓存

CUDA Graph 对 GEMM 还有一个常被忽略的**正面影响**。

### 10.1 cuBLAS 的算法选择机制

cuBLAS 内部有多种 GEMM 算法（不同的 tiling、pipeline 策略）。每次调用时，cuBLAS 会根据 M/N/K 选择"最优"算法：

```
cublasLtMatmul():
  1. 检查 (M, N, K, dtype) → heuristic cache
  2. 如果 cache hit → 直接用缓存的最优算法
  3. 如果 cache miss → 运行 heuristic 选择算法
```

### 10.2 CUDA Graph 的加速效果

没有 CUDA Graph 时：M 是任意值，cache miss 率高，heuristic 选择有开销。

有 CUDA Graph 时：M 只有 ~50 个值，且在 warmup/capture 阶段已经调用过，cache 全部 warm。运行时 **100% cache hit**。

### 10.3 对 Triton 的类似效果

Triton 的 `@autotune` 也有类似的行为：

```
autotune cache:
  (M=1, N=22016, K=4096) → Config(BLOCK_M=32, BLOCK_N=64, BLOCK_K=64)
  (M=8, N=22016, K=4096) → Config(BLOCK_M=64, BLOCK_N=64, BLOCK_K=32)
  (M=32, N=22016, K=4096) → Config(BLOCK_M=128, BLOCK_N=128, BLOCK_K=32)
  ...
```

因为 M 被限制到有限值，autotune cache 不会无限增长，而且每个值都在 capture 阶段就 warm 好了。

## 11. CUDA Graph 的显存成本

CUDA Graph 不是免费的——每个录制好的 graph 占用显存。

In [ ]:
# ============================================================
# 估算 CUDA Graph 的显存消耗
# ============================================================

def estimate_cudagraph_memory(
    num_layers: int,
    hidden_size: int,
    intermediate_size: int,
    num_heads: int,
    head_dim: int,
    capture_sizes: list,
    dtype_bytes: int = 2,  # FP16
):
    """
    估算 CUDA Graph 的额外显存消耗。

    CUDA Graph 需要保存所有中间激活值（因为地址必须固定）。
    每个 capture size 都需要自己的一套激活值缓冲区。
    """
    results = []

    # 去重 capture sizes
    unique_sizes = sorted(set(capture_sizes))

    for M in unique_sizes:
        # 每层的中间激活值（粗略估算）
        per_layer_bytes = 0
        # QKV 输出: M × (num_heads × head_dim × 3)
        per_layer_bytes += M * num_heads * head_dim * 3 * dtype_bytes
        # Attention 输出: M × (num_heads × head_dim)
        per_layer_bytes += M * num_heads * head_dim * dtype_bytes
        # MLP gate_up 输出: M × intermediate_size × 2
        per_layer_bytes += M * intermediate_size * 2 * dtype_bytes
        # MLP down 输出: M × hidden_size
        per_layer_bytes += M * hidden_size * dtype_bytes
        # Residual connections etc.
        per_layer_bytes += M * hidden_size * 2 * dtype_bytes

        total_bytes = per_layer_bytes * num_layers
        results.append((M, total_bytes))

    return results, unique_sizes


# Qwen3-8B 参数
results, sizes = estimate_cudagraph_memory(
    num_layers=32,
    hidden_size=4096,
    intermediate_size=11008,
    num_heads=32,
    head_dim=128,
    capture_sizes=compute_capture_sizes(512),
)

total_memory = sum(b for _, b in results)

print(f"Qwen3-8B, FP16, capture sizes up to 512")
print(f"去重后的 capture sizes 数量: {len(sizes)}")
print()
print(f"{'M':>6} {'显存 (MB)':>12}")
print("-" * 20)
for M, mem_bytes in results[::10]:  # 每隔 10 个打印一个
    print(f"{M:>6} {mem_bytes / 1024**2:>10.1f} MB")
print(f"{'...':>6}")
print(f"{'合计':>6} {total_memory / 1024**3:>10.2f} GB")
print()

# 但实际上 vLLM 使用 graph_pool_handle 来共享内存！
print("注意: vLLM 使用 graph_pool_handle() 让不同 size 的 graph 共享 GPU 内存池")
print("实际显存 ≈ 最大 capture size 的激活值大小（不是所有 size 的总和）")
max_mem = results[-1][1]
print(f"实际显存 ≈ {max_mem / 1024**2:.1f} MB (最大 size={sizes[-1]} 的激活值)")

## 12. 完整流程图

### Decode 请求在 FULL_DECODE_ONLY 模式下的 GEMM 路径

```
用户请求 (5 个 token 待生成)
          │
          ▼
┌─────────────────────────────┐
│  Scheduler: batch = 5 tokens │
└─────────────────────────────┘
          │
          ▼
┌─────────────────────────────┐
│  padding_map[5] = 8          │
│  pad input_ids to 8          │
└─────────────────────────────┘
          │
          ▼
┌─────────────────────────────┐
│  graphs[8].replay()          │  ← 一次 API 调用
│                               │
│  ┌── Layer 0 ──────────────┐ │
│  │ LayerNorm(M=8, K=4096)  │ │
│  │ QKV GEMM(M=8, K=4096,   │ │
│  │          N=6144)         │ │  ← K, N 是常量; M=8 是此 graph 的固定值
│  │ Attention(8 tokens)      │ │
│  │ O GEMM(M=8, K=4096,     │ │
│  │         N=4096)          │ │
│  │ MLP gate_up GEMM(M=8,   │ │
│  │    K=4096, N=22016)      │ │  ← 这就是核心 GEMM
│  │ SiLU+Mul                 │ │
│  │ MLP down GEMM(M=8,      │ │
│  │    K=11008, N=4096)      │ │
│  └──────────────────────────┘ │
│  ┌── Layer 1 ... Layer 31 ──┐ │
│  │        (同上)              │ │
│  └──────────────────────────┘ │
└─────────────────────────────┘
          │
          ▼
┌─────────────────────────────┐
│  hidden_states[:5]           │  ← 截断 padding 部分
│  output shape: (5, 4096)     │
└─────────────────────────────┘
```

## 源码映射表

| 本 Notebook 概念 | vLLM 源码位置 | 说明 |
|-----------------|-------------|------|
| CUDAGraphMode 枚举 | `vllm/config/compilation.py:51-61` | NONE/PIECEWISE/FULL/FULL_DECODE_ONLY |
| capture sizes 计算 | `vllm/config/vllm.py:1269-1281` | [1,2,4,8,16,...] 默认逻辑 |
| padding map 构建 | `vllm/v1/worker/gpu/cudagraph_utils.py:298-331` | actual→padded 映射 |
| CudaGraphManager | `vllm/v1/worker/gpu/cudagraph_utils.py:27-295` | 管理录制/回放 |
| _capture_full_graph | `vllm/v1/worker/gpu/cudagraph_utils.py:158-193` | FULL 模式录制 |
| _capture_piecewise_graph | `vllm/v1/worker/gpu/cudagraph_utils.py:195-225` | PIECEWISE 模式录制 |
| run_fullgraph (replay) | `vllm/v1/worker/gpu/cudagraph_utils.py:291-295` | 回放 + 截断 |
| get_cudagraph_runtime_mode | `vllm/v1/worker/gpu/cudagraph_utils.py:275-289` | 运行时模式选择 |
| dynamic_arg_dims | `vllm/compilation/decorators.py:110-232` | torch.compile 动态维度标记 |
| Qwen3_5Model compile 标注 | `vllm/model_executor/models/qwen3_5.py:290-298` | input_ids dim 0 是动态 |
| graph_pool_handle | `vllm/v1/worker/gpu/cudagraph_utils.py:64` | 共享 GPU 内存池 |
| warmup before capture | `vllm/v1/worker/gpu/cudagraph_utils.py:128-141` | 确保 JIT 编译先完成 |

## 本章总结

### 核心结论

**Q: CUDA Graph `FULL_DECODE_ONLY` 下，GEMM 的 M/N/K 是编译时常量吗？**

**A: N 和 K 是编译时常量（由模型结构决定），M 不是常量但被约束到 ~50 个离散值。**

详细来说：

| 维度 | 是否常量 | 来源 | 值的范围 |
|------|---------|------|---------|
| K | ✅ 编译时常量 | `hidden_size` | 4096 (Qwen3-8B) |
| N | ✅ 编译时常量 | `intermediate_size` | 22016 (gate_up) |
| M | ❌ 运行时离散 | `num_tokens` | [1,2,4,8,...,512] |

### CUDA Graph 对 GEMM 的五个影响

1. **M 被离散化** — 从任意正整数变成 ~50 个 capture sizes
2. **Padding 浪费** — 小 batch 浪费率高（M=5→8, 37%），大 batch 可忽略（M=250→256, 2%）
3. **消除 launch overhead** — 352 次 kernel launch → 1 次 graph replay
4. **cuBLAS/Triton autotune 缓存效率高** — 只有 ~50 种 M 值，cache 100% warm
5. **显存开销** — graph_pool 共享内存，实际只占最大 capture size 的激活值大小

### 对自定义 Triton GEMM 的建议

1. 确保 kernel 在 warmup 阶段完成 JIT 编译和 autotune
2. `@triton.autotune(key=['M','N','K'])` 会自动适配离散 M 值
3. 不要在 kernel 内部做 CPU 同步操作
4. 使用显式 pointer arithmetic 而非 `make_block_ptr`（兼容性更好）

---
← [Notebook 11: 在 Qwen3.5 上部署自定义 Triton GEMM](11-custom-triton-gemm-on-qwen.ipynb)